# Exercise 4: Twiss parameters from the one-turn matrix

**Block 2: Twiss, ellipses and emittance**

**Learning objectives**
* Extract $\beta$, $\alpha$, $\gamma$ and the phase advance $\mu$ from a one-turn matrix.
* Check that any stable one-turn matrix can be written as $M=\cos\mu I +\sin\mu J$.
* Propagate $\beta(s)$ around the ring in two ways and see that they agree.

**Motivation.** The one-turn matrix tells you everything about the *linear* motion in a ring, but
in a disguised form. The Twiss parameters are the "decoded" version: $\beta$ tells you how big the beam
is at each place, $\alpha$ whether it is converging or diverging, $\mu$ how fast particles rotate in phase space.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## The machine: a small FODO ring

From now on we use a small ring (`beamlib.lattices.fodo_ring`) made of 8 identical FODO cells,
each 6 m long, with thick quadrupoles and two sector dipoles per cell (16 dipoles × 22.5° = 360°).
The ring starts in the middle of a focusing quadrupole.

`O.one_turn_matrix(ring)` multiplies all element matrices. It returns a 5×5 matrix acting on
($x$, $p_x$, $y$, $p_y$, $\delta$). Here we only need the horizontal 2×2 block `M[:2, :2]` and the
vertical one `M[2:4, 2:4]`.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

ring = L.fodo_ring()
M = O.one_turn_matrix(ring)
Mx, My = M[:2, :2], M[2:4, 2:4]
print("circumference:", sum(el.length for el in ring), "m")
print("Mx =\n", Mx.round(4))
print("My =\n", My.round(4))

## 1. Decoding the one-turn matrix

Any 2×2 one-turn matrix with det = 1 and $|\text{Tr}(M)| < 2$ can be written as

$$M = \cos\mu\, I + \sin\mu\, J, \qquad J = \begin{pmatrix} \alpha & \beta \\ -\gamma & -\alpha\end{pmatrix},
\qquad \beta\gamma - \alpha^2 = 1 .$$

Reading off the elements:
* $\cos\mu = (M_{11} + M_{22})/2$
* $\beta = M_{12} / \sin\mu$, $\alpha = (M_{11} − M_{22}) / (2\sin\mu)$,  $\gamma = −M_{21} / \sin\mu$

The sign of $\sin\mu$ is not fixed by $\cos\mu$. We choose it so that $\beta > 0$ ($\beta$ is a size, it must be positive).

**Task.** Implement `twiss_from_one_turn(M2)` and return $\mu$ in $[0, 2\pi)$.

In [ ]:
def twiss_from_one_turn(M2: np.ndarray) -> dict:
    """beta [m], alpha, gamma [1/m] and mu [rad, in 0..2pi] of a 2x2 one-turn matrix."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
for name, M2 in [("x", Mx), ("y", My)]:
    tw = twiss_from_one_turn(M2)
    ref = O.twiss_from_matrix(M2)
    for key in ("beta", "alpha", "gamma", "mu"):
        assert np.isclose(tw[key], ref[key], atol=1e-12), f"{key} wrong in {name}"
    # rebuild M from the Twiss parameters
    J = np.array([[tw["alpha"], tw["beta"]], [-tw["gamma"], -tw["alpha"]]])
    assert np.allclose(np.cos(tw["mu"]) * np.eye(2) + np.sin(tw["mu"]) * J, M2), "M = cos mu I + sin mu J fails"
    assert np.isclose(tw["beta"] * tw["gamma"] - tw["alpha"] ** 2, 1.0)
    print(f"{name}: beta = {tw['beta']:.4f} m, alpha = {tw['alpha']:.2e}, "
          f"gamma = {tw['gamma']:.4f} 1/m, mu/2pi = {tw['mu'] / (2 * np.pi):.5f}")
print("GOOD! M = cos(mu) I + sin(mu) J verified in both planes")

$\alpha\approx0$ here because the ring starts in the middle of a quadrupole, a symmetry point where $\beta$ has
a maximum (in $x$) or minimum (in $y$).

Note that $\mu/2\pi$ is only the **fractional part** of the tune: the matrix cannot know how many full
oscillations happened during the turn (cos and sin are periodic). We will get the integer part below.

## 2. Propagating $\beta(s)$ around the ring

If we know ($\beta_0$, $\alpha_9$) at $s = 0$ and the matrix $M(s)$ from 0 to $s$, the Twiss parameters at $s$ follow.
Two equivalent ways:

**(a) Direct matrix propagation.** The matrix $B = [[\beta, −\alpha], [−\alpha, \gamma]]$ is transported like
$B(s) = M B_0 M^T$. (This is `O.propagate_twiss`.)

**(b) The ($\beta$, $\alpha$, $\Delta\mu$) form.** Any transfer matrix between two points can be written using only the
Twiss parameters at both ends and the phase advance $\Delta\mu$ between them (known formula,
`O.transfer_matrix_from_twiss`). Inverting it gives

$$\beta(s) = \frac{(M_{11}\beta_0 - M_{12}\alpha_0)^2 + M_{12}^2}{\beta_0},\qquad
\tan\Delta\mu = \frac{M_{12}}{M_{11}\beta_0 - M_{12}\alpha_0} .$$

**Task.** Implement (b) in `beta_and_phase(M2, beta0, alpha0)`, returning $\beta(s)$ and $\Delta\mu$ (use `np.arctan2`
and wrap $\Delta\mu$ into $[0, 2\pi)$).

In [ ]:
def beta_and_phase(M2: np.ndarray, beta0: float, alpha0: float):
    """beta at the end of the 2x2 matrix M2 and the phase advance through it [rad]."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
twx = twiss_from_one_turn(Mx)
s, elem, Ms = O.cumulative_matrices(ring, n_slices=20)

beta_b = np.array([beta_and_phase(Mj[:2, :2], twx["beta"], twx["alpha"])[0] for Mj in Ms])
dmu_b = np.array([beta_and_phase(Mj[:2, :2], twx["beta"], twx["alpha"])[1] for Mj in Ms])
beta_a = np.array([O.propagate_twiss(Mj[:2, :2], twx["beta"], twx["alpha"])[0] for Mj in Ms])
alpha_a = np.array([O.propagate_twiss(Mj[:2, :2], twx["beta"], twx["alpha"])[1] for Mj in Ms])

assert np.allclose(beta_a, beta_b, rtol=1e-10), "the two methods should give the same beta"
# the (beta, alpha, dmu) form must rebuild every transfer matrix
for j in range(0, len(s), 37):
    Mrebuilt = O.transfer_matrix_from_twiss(twx["beta"], twx["alpha"], beta_b[j], alpha_a[j], dmu_b[j])
    assert np.allclose(Mrebuilt, Ms[j][:2, :2], atol=1e-10)
assert np.isclose(beta_b[-1], twx["beta"]), "after one turn beta must come back (periodicity)"
print("GOOD! both propagation methods agree, and beta is periodic")

**Predict first.** Where in the cell will $\beta_x$ be largest, and where $\beta_y$? How many times
does $\beta_x$ go up and down around the ring?

*Your prediction:*

In [ ]:
tw = O.twiss_along(ring, twx["beta"], twx["alpha"], O.twiss_from_matrix(My)["beta"],
                   O.twiss_from_matrix(My)["alpha"], n_slices=20)

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(tw["s"], tw["betx"], color=COLORS["x"], label=r"$\beta_x$")
axs[0].plot(tw["s"], tw["bety"], color=COLORS["y"], label=r"$\beta_y$")
axs[0].plot(s[::20], beta_b[::20], "b.", ms=4, label=r"$\beta_x$ from the $(\beta,\alpha,\Delta\mu)$ form")
for i, el in enumerate(ring):                                   # draw the magnets
    s0 = sum(e.length for e in ring[:i])
    if isinstance(el, E.Quadrupole):
        axs[0].axvspan(s0, s0 + el.length, color=COLORS["x"] if el.k > 0 else COLORS["y"], alpha=0.2, lw=0)
    elif isinstance(el, E.SectorBend):
        axs[0].axvspan(s0, s0 + el.length, color="0.8", alpha=0.5, lw=0)
axs[0].set_ylabel("β [m]"); axs[0].legend(loc="upper right")
axs[0].set_title("Optics of the FODO ring (blue band: QF, red: QD, grey: dipoles)")
axs[1].plot(tw["s"], tw["mux"] / (2 * np.pi), color=COLORS["x"], label=r"$\mu_x / 2\pi$")
axs[1].plot(tw["s"], tw["muy"] / (2 * np.pi), color=COLORS["y"], label=r"$\mu_y / 2\pi$")
axs[1].set_xlabel("s [m]"); axs[1].set_ylabel("phase advance / 2π"); axs[1].legend()
plt.show()

## 3. The full tune

Adding up the phase advances of all the pieces gives the **total** phase advance over one turn,
including the integer part.

**Task.** Compute the total tunes $Q_x$, $Q_y$ from `tw["mux"]` and `tw["muy"]`, and check that their
fractional parts agree with $\mu/2\pi$ from the one-turn matrix.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
print(f"Qx = {Qx:.5f}, Qy = {Qy:.5f}")
assert np.isclose(Qx % 1, twx["mu"] / (2 * np.pi)) and np.isclose(Qy % 1, O.twiss_from_matrix(My)["Q"])
print("GOOD! Fractional parts agree with the one-turn matrix")

## 4. Cell vs ring

The ring is 8 identical cells. Compute the Twiss parameters from the matrix of **one cell**
(the first 12 elements). How do they compare with the ring's, and how is the phase advance per
cell related to the tune?

In [ ]:
M_cell = O.one_turn_matrix(ring[:12])
tw_cell = twiss_from_one_turn(M_cell[:2, :2])
print(f"cell: beta_x = {tw_cell['beta']:.4f} m, mu_x = {np.degrees(tw_cell['mu']):.3f} deg")
print(f"ring: beta_x = {twx['beta']:.4f} m, 8 * mu_cell / 2pi = {8 * tw_cell['mu'] / (2 * np.pi):.5f} = Qx")
assert np.isclose(tw_cell["beta"], twx["beta"]) and np.isclose(8 * tw_cell["mu"] / (2 * np.pi), Qx)

## Reflection questions

1. $\beta$ and $\alpha$ do not depend on the particle at all. What *does* depend on the particle?
2. Why must $\beta$ come back to the same value after one turn, while the phase $\mu$ keeps growing?
3. The ring tune is 8 x (phase advance per cell). Why can you not get the integer part of the tune
   from the one-turn matrix alone?